Lable Encoding

- Goal  - turn categorical (non-numeric) data into numbers .
-  ML models eat numbers and produce numbers.

Encoding Technique

  - Label encoding
  - ordinal encoding
  - one- hot encoding
  - binary encoding
  - Target(mean) encoding
  - Frequency / count encoding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing  import LabelEncoder, OrdinalEncoder, OneHotEncoder
pd.set_option('display.max_columns',None)


# Toy Dataset

  - color     - nominal  - no order
  - size      - ordinal   S < M < L < XL
  - city      - nominal but high cardinality - many possible values
  - purchase   -1/0

In [ ]:
df = pd.DataFrame({
    "color": ['Red','Green','Blue','Green','Red','Blue','Red','Green'],
    'size' :  ['S','L','M','XL','M','S','L','M'],
    'city' :  ['Mumbai','Delhi','Chennai','Mumbai','Delhi','Mumbai','Chennai','Mumbai'],

    'purchased':[1,0,1,0,1,1,0,1]
})

df

,color,size,city,purchased
0,Red,S,Mumbai,1
1,Green,L,Delhi,0
2,Blue,M,Chennai,1
3,Green,XL,Mumbai,0
4,Red,M,Delhi,1
5,Blue,S,Mumbai,1
6,Red,L,Chennai,0
7,Green,M,Mumbai,1


In [ ]:
print('No of rows and columns',df.shape[0],df.shape[1])

No of rows and columns 8 4


In [ ]:
for column in df.columns:
  print('Number of unique values' , column,  df[column].nunique())

Number of unique values color 3
Number of unique values size 4
Number of unique values city 3
Number of unique values purchased 2


In [ ]:
for column in df.columns:
  print(' unique values' , column,  df[column].unique())

 unique values color ['Red' 'Green' 'Blue']
 unique values size ['S' 'L' 'M' 'XL']
 unique values city ['Mumbai' 'Delhi' 'Chennai']
 unique values purchased [1 0]


1 . Label Encoding

##### it is simplest technique where each unique category is assigned a integer (usually alphabetically)

    -  Blue -> 0  , Green -> 1 , Red -> 2
    - this fast and memory light - it stays in single column

In [ ]:
le = LabelEncoder()
df_label = df.copy()
df_label['encoded_color']= le.fit_transform(df['color'])
# see mapping the  encoder chose
print('Label encoded classes',le.classes_)
mapping = dict (zip(le.classes_, le.transform(le.classes_)))
print(mapping)
df_label[['color','encoded_color']]

Label encoded classes ['Blue' 'Green' 'Red']
{'Blue': np.int64(0), 'Green': np.int64(1), 'Red': np.int64(2)}


,color,encoded_color
0,Red,2
1,Green,1
2,Blue,0
3,Green,1
4,Red,2
5,Blue,0
6,Red,2
7,Green,1


The trap : accidental order

- the model see number and  assumes red(2)  > geen (1) > blue (0)
- for color it doesnt make sense .Giving this to linear or neural network inject
  false ranking
#### Safe to use when
     -  its encoding  the target (y)  
    - if tree based models (decision tree , random forest , xgboost etc)
     they split data on threshold and dont care about fake ordering
    
- avoid for nominal features going to linear model / neural network. Use one hot encoding

# 2 Ordinal Encoding
   - it is similar to label encoding - one column of integers
   - here the order is intentional and real
   - define the sequence instead of letting the encoder guess alphabetically

   - size is ordered  S < M < L < XL

In [ ]:
size_orders=["S","M","L","XL"]  # WE decide meaniful order
oe = OrdinalEncoder(categories=[size_orders])
df_ord = df.copy()
df_ord["encoded_size"]= oe.fit_transform(df[['size']]).astype(int)
df_ord

,color,size,city,purchased,encoded_size
0,Red,S,Mumbai,1,0
1,Green,L,Delhi,0,2
2,Blue,M,Chennai,1,1
3,Green,XL,Mumbai,0,3
4,Red,M,Delhi,1,1
5,Blue,S,Mumbai,1,0
6,Red,L,Chennai,0,2
7,Green,M,Mumbai,1,1


In [ ]:
df_ord[['size','encoded_size']].drop_duplicates().sort_values('encoded_size')

,size,encoded_size
0,S,0
2,M,1
1,L,2
3,XL,3


#  3. One Hot Encoding
   
    - this cure the false order problem.
    - Instead of one numbered column
    - It will create a separate binary(0/1) column per category
    - No category is bigger than another
    - Each category get a yes/no flag

In [ ]:
# through pandas
one_hot_encoding = pd.get_dummies(df.color,prefix='color').astype(int)
one_hot_encoding

,color_Blue,color_Green,color_Red
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0
4,0,0,1
5,1,0,0
6,0,0,1
7,0,1,0


#### Dummy encoding - drop one column

In [ ]:
one_hot_encoding1 = pd.get_dummies(df.color,prefix='color',drop_first=True).astype(int)
one_hot_encoding1

,color_Green,color_Red
0,0,1
1,1,0
2,0,0
3,1,0
4,0,1
5,0,0
6,0,1
7,1,0


In [ ]:
# the scikit way
encoder =OneHotEncoder(sparse_output=False,dtype=int)
arr = encoder.fit_transform(df[['color']])
pd.DataFrame(arr,columns=encoder.get_feature_names_out(['color']))

,color_Blue,color_Green,color_Red
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0
4,0,0,1
5,1,0,0
6,0,0,1
7,0,1,0


## The trap :  dimensionality explosion

  -  A column with 1000 unique values becomes 1000 new columns
  -  training slow and model overfit. this is the curse of dimensionality
  
- Use one hot encoding when the no of categories is small  - (rule of thumb - under 10 to 15)

- For higher cadinality feature use Binary or frequency encoding
